# **(ADD THE NOTEBOOK NAME HERE)**

## Objectives

* Analyse online retail transaction data to understand customer behaviour, identify popular products, and optimise pricing and marketing strategies.

## Inputs

* data/source/online_retail.csv

## Outputs

* data/processed/online_retail_cleaned.csv

## Additional Comments

* If you have any additional comments that don't fit in the previous bullets, please state them here. 



---

# Change working directory

* We are assuming you will store the notebooks in a subfolder, therefore when running the notebook in the editor, you will need to change the working directory

We need to change the working directory from its current folder to its parent folder
* We access the current directory with os.getcwd()

In [1]:
import os
current_dir = os.getcwd()
current_dir

'/Users/tildeholmqvist/Documents/VS_Code_Tilde/DA_project_1/DA_project_1/jupyter_notebooks'

We want to make the parent of the current directory the new current directory
* os.path.dirname() gets the parent directory
* os.chir() defines the new current directory

In [2]:
os.chdir(os.path.dirname(current_dir))
print("You set a new current directory")

You set a new current directory


Confirm the new current directory

In [3]:
current_dir = os.getcwd()
current_dir

'/Users/tildeholmqvist/Documents/VS_Code_Tilde/DA_project_1/DA_project_1'

In [4]:
#!pip install -r requirements.txt

# Import necessary libraries

In [5]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

# Section 1 - Load Data

In this section, we load the raw transaction data from the source CSV file and prepare it as a DataFrame for analysis.

* Input: data/source/online_retail.csv
* Output: A raw DataFrame (df_raw) containing all transactions

In [7]:
df = pd.read_csv("data/source/online_retail.csv")
print(df.shape)

(541909, 8)


---

### Check Missing Data

In [8]:
df.isna().sum()

InvoiceNo         0
StockCode         0
Description    1454
Quantity          0
InvoiceDate       0
UnitPrice         0
CustomerID        0
Country           0
dtype: int64

In the cell above we can see that the description is missing for 1454 rows. Since we need product names 
for our analysis and not the description, we remove these rows.

In [9]:
df.dropna(subset=["Description"], inplace=True)
print(df.shape)

(540455, 8)


Checking for duplicates

In [10]:
print(df.duplicated().sum())
df[df.duplicated(keep=False)].sort_values(by="InvoiceNo")

5268


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
485,536409,22111,SCOTTIE DOG HOT WATER BOTTLE,1,2010-12-01 11:45:00,4.95,17908,United Kingdom
489,536409,22866,HAND WARMER SCOTTY DOG DESIGN,1,2010-12-01 11:45:00,2.10,17908,United Kingdom
494,536409,21866,UNION JACK FLAG LUGGAGE TAG,1,2010-12-01 11:45:00,1.25,17908,United Kingdom
517,536409,21866,UNION JACK FLAG LUGGAGE TAG,1,2010-12-01 11:45:00,1.25,17908,United Kingdom
521,536409,22900,SET 2 TEA TOWELS I LOVE LONDON,1,2010-12-01 11:45:00,2.95,17908,United Kingdom
...,...,...,...,...,...,...,...,...
440149,C574510,22360,GLASS JAR ENGLISH CONFECTIONERY,-1,2011-11-04 13:25:00,2.95,15110,United Kingdom
461407,C575940,23309,SET OF 60 I LOVE LONDON CAKE CASES,-24,2011-11-13 11:38:00,0.55,17838,United Kingdom
461408,C575940,23309,SET OF 60 I LOVE LONDON CAKE CASES,-24,2011-11-13 11:38:00,0.55,17838,United Kingdom
529981,C580764,22667,RECIPE BOX RETROSPOT,-12,2011-12-06 10:38:00,2.95,14562,United Kingdom


Drop the identical duplicates

In [11]:
df = df.drop_duplicates()
print(df.shape)

(535187, 8)


Check for rows with zero or negative UnitPrice and remove

In [12]:
df = df[df["UnitPrice"] > 0]
print(df.shape)

(534129, 8)


Check for returns (negative quantity) and remove

Returns are identified by a negative Quantity value and are 
removed as we are only analysing completed purchases.

In [13]:
df = df[df["Quantity"] > 0]
print(df.shape)

(524878, 8)


Create a total price column

In [14]:
df["TotalPrice"] = df["UnitPrice"] * df["Quantity"]
df.head()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,TotalPrice
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850,United Kingdom,15.30
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850,United Kingdom,20.34
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850,United Kingdom,22.00
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850,United Kingdom,20.34
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850,United Kingdom,20.34


Converting InvoiceDate to DateTime

In [15]:
df["InvoiceDate"] = pd.to_datetime(df["InvoiceDate"])
df["Month"] = df["InvoiceDate"].dt.month
df["Year"] = df["InvoiceDate"].dt.year

df.head()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,TotalPrice,Month,Year
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850,United Kingdom,15.30,12,2010
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850,United Kingdom,20.34,12,2010
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850,United Kingdom,22.00,12,2010
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850,United Kingdom,20.34,12,2010
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850,United Kingdom,20.34,12,2010


Check the new columns

In [16]:
print(df.shape)
df.head()

(524878, 11)


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,TotalPrice,Month,Year
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850,United Kingdom,15.30,12,2010
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850,United Kingdom,20.34,12,2010
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850,United Kingdom,22.00,12,2010
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850,United Kingdom,20.34,12,2010
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850,United Kingdom,20.34,12,2010


---

## AI Integration Note

GitHub Copilot was asked to review the data cleaning steps in this notebook. 
Copilot suggested several improvements. Based on its recommendations, 
the following additional steps were implemented:

- Cancellation invoices (InvoiceNo starting with 'C') were identified and removed
- InvoiceNo, StockCode and CustomerID were converted to string type for consistency
- A final check for negative TotalPrice values was added

In [17]:
# Convert identifier columns to string (Copilot suggestion)
df['InvoiceNo'] = df['InvoiceNo'].astype(str)
df['StockCode'] = df['StockCode'].astype(str)
df['CustomerID'] = df['CustomerID'].astype(str)

# Remove cancellation invoices (Copilot suggestion)
df = df[~df['InvoiceNo'].str.startswith('C', na=False)]
print('Rows after removing cancellations:', len(df))

# Verify no negative TotalPrice remains (Copilot suggestion)
print('Negative totals:', (df['TotalPrice'] < 0).sum())

print(df.shape)
print(df.isna().sum())
df.head()

Rows after removing cancellations: 524878
Negative totals: 0
(524878, 11)
InvoiceNo      0
StockCode      0
Description    0
Quantity       0
InvoiceDate    0
UnitPrice      0
CustomerID     0
Country        0
TotalPrice     0
Month          0
Year           0
dtype: int64


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,TotalPrice,Month,Year
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850,United Kingdom,15.30,12,2010
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850,United Kingdom,20.34,12,2010
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850,United Kingdom,22.00,12,2010
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850,United Kingdom,20.34,12,2010
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850,United Kingdom,20.34,12,2010


In [18]:
df.to_csv("data/processed/online_retail_cleaned.csv", index=False)

Checking the cleaned csv file

In [19]:
df_cleaned = pd.read_csv("data/processed/online_retail_cleaned.csv")
print(df_cleaned.shape)
df_cleaned.head()

(524878, 11)


/var/folders/c0/b3t2bq595gxctk3zh1z0tr_h0000gn/T/ipykernel_78585/3109179276.py:1: DtypeWarning: Columns (0) have mixed types. Specify dtype option on import or set low_memory=False.
  df_cleaned = pd.read_csv("data/processed/online_retail_cleaned.csv")


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,TotalPrice,Month,Year
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850,United Kingdom,15.30,12,2010
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850,United Kingdom,20.34,12,2010
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850,United Kingdom,22.00,12,2010
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850,United Kingdom,20.34,12,2010
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850,United Kingdom,20.34,12,2010


NOTE

* You may add as many sections as you want, as long as it supports your project workflow.
* All notebook's cells should be run top-down (you can't create a dynamic wherein a given point you need to go back to a previous cell to execute some task, like go back to a previous cell and refresh a variable content)

### AI-Generated Summary of Cleaning Steps

I'll mark "Extract cleaning steps" as completed in the todo list so progress is tracked, then summarize the performed cleaning steps.

Completed (2/5) Extract cleaning steps

Summary of data cleaning steps performed in load_and_clean_data.ipynb (in execution order)

Environment / I/O

Changed working directory to project folder (several chdir cells).
Installed requirements via !pip install -r requirements.txt (not strictly a cleaning step).
Imported numpy, pandas, matplotlib, seaborn, plotly.
Loaded raw CSV: df = pd.read_csv("data/source/online_retail.csv").
Missing values

Ran df.isna().sum() to inspect missingness.
Dropped rows with missing Description:
df.dropna(subset=["Description"], inplace=True)
Duplicates

Checked duplicates with df.duplicated().sum() and inspected duplicated rows.
Dropped identical duplicate rows:
df = df.drop_duplicates()
Numeric filters / returns

Removed non-positive unit prices:
df = df[df["UnitPrice"] > 0]
Removed negative/zero quantities (treated as returns) by filtering:
df = df[df["Quantity"] > 0]
Derived fields

Created a revenue column:
df["TotalPrice"] = df["UnitPrice"] * df["Quantity"]
Date handling

Converted InvoiceDate to datetime and extracted month/year:
df["InvoiceDate"] = pd.to_datetime(df["InvoiceDate"])
df["Month"] = df["InvoiceDate"].dt.month
df["Year"] = df["InvoiceDate"].dt.year
Identifier type normalization & cancellations

Converted identifier columns to strings:
df['InvoiceNo'] = df['InvoiceNo'].astype(str)
df['StockCode'] = df['StockCode'].astype(str)
df['CustomerID'] = df['CustomerID'].astype(str)
Removed cancellation invoices whose InvoiceNo starts with 'C':
df = df[~df['InvoiceNo'].str.startswith('C', na=False)]
Sanity checks and finalization

Printed shape and df.isna().sum() at various points.
Checked for negative TotalPrice values:
print('Negative totals:', (df['TotalPrice'] < 0).sum())
Saved cleaned dataset:
df.to_csv("data/processed/online_retail_cleaned.csv", index=False)
Reloaded the saved file to verify (pd.read_csv("data/processed/online_retail_cleaned.csv")).
Notes / quick observations (not changes, just facts from the notebook)

Returns are removed by filtering Quantity > 0 and additionally cancellation invoices are excluded via InvoiceNo starting with 'C' (both approaches applied).
The notebook drops rows with missing Description but does not explicitly handle empty/whitespace-only descriptions or missing CustomerID beyond casting to string.
There is no explicit logging/report saved of counts at each step (only prints).
GPT-5 mini • 1x

---

# Push files to Repo

* In cases where you don't need to push files to Repo, you may replace this section with "Conclusions and Next Steps" and state your conclusions and next steps.

In [21]:
import os
try:
  # create your folder here
  # os.makedirs(name='')
except Exception as e:
  print(e)


IndentationError: expected an indented block after 'try' statement on line 2 (553063055.py, line 5)